# DESI QSO Spectra — PCA Decomposition

Reproduces Figures 7.4, 7.5 and 7.6 from *Statistics, Data Mining, and Machine Learning
in Astronomy* (Ivezić et al.) using DESI DR1 QSO spectra instead of SDSS.

- **Fig 7.4**: PCA components (mean + eigenvectors)
- **Fig 7.5**: Eigenvalues and cumulative explained variance
- **Fig 7.6**: Spectrum reconstruction with increasing number of components

Load matrices from `desi_spectra_prep.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import os

os.makedirs('plots', exist_ok=True)

## 1. Load data

In [ ]:
# High-z QSOs (z>2): rest-frame UV, Ly-α region
grid_hi   = np.load('data/qso_highz_grid.npy')
matrix_hi = np.load('data/qso_highz_matrix.npy')
z_hi      = np.load('data/qso_highz_z.npy')

# Low-z QSOs (0.5<z<1): rest-frame optical
grid_lo   = np.load('data/qso_lowz_grid.npy')
matrix_lo = np.load('data/qso_lowz_matrix.npy')
z_lo      = np.load('data/qso_lowz_z.npy')

print(f'High-z: {matrix_hi.shape}  z range: {z_hi.min():.2f}–{z_hi.max():.2f}')
print(f'Low-z:  {matrix_lo.shape}  z range: {z_lo.min():.2f}–{z_lo.max():.2f}')

## 2. Fit PCA

We fit PCA on the **mean-subtracted** spectra. The mean spectrum itself
is the zeroth component (following the astroML convention).

In [ ]:
N_COMPONENTS = 20   # total eigenvectors to compute

def fit_pca(matrix, n_components=N_COMPONENTS):
    mean   = matrix.mean(axis=0)
    pca    = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
    pca.fit(matrix)
    evals  = pca.explained_variance_ratio_          # normalized eigenvalues
    evals_cs = np.cumsum(evals)                     # cumulative
    evecs  = pca.components_                        # shape (n_components, n_pixels)
    return mean, evecs, evals, evals_cs

mean_hi, evecs_hi, evals_hi, evals_cs_hi = fit_pca(matrix_hi)
mean_lo, evecs_lo, evals_lo, evals_cs_lo = fit_pca(matrix_lo)

print(f'High-z: top-5 eigenvalues = {evals_hi[:5].round(4)}')
print(f'        variance with 10 components = {evals_cs_hi[9]:.3f}')
print(f'Low-z:  top-5 eigenvalues = {evals_lo[:5].round(4)}')
print(f'        variance with 10 components = {evals_cs_lo[9]:.3f}')

## 3. Figure 7.4 — PCA components

Mean spectrum + first 4 eigenvectors for each sample.
Reproduces Fig 7.4 of Ivezić et al. (left column, PCA only).

In [ ]:
def plot_components(grid, mean, evecs, evals, title, fname, n_show=5):
    """
    Fig 7.4: mean + first n_show-1 PCA eigenvectors.
    Components are stacked vertically; zero line shown in gray.
    """
    comps  = np.vstack([mean, evecs[:n_show-1]])   # (n_show, n_pix)
    labels = ['mean'] + [f'component {i+1}' for i in range(n_show-1)]

    fig, axes = plt.subplots(n_show, 1, figsize=(10, 2*n_show), sharex=True)
    fig.subplots_adjust(hspace=0.05)

    for i, (ax, comp, lbl) in enumerate(zip(axes, comps, labels)):
        ax.plot(grid, comp, 'k', lw=1.0)
        ax.axhline(0, color='gray', lw=0.8, ls='--')
        ax.set_ylabel(r'$F_\lambda$', fontsize=9)
        ax.text(0.02, 0.88, lbl, transform=ax.transAxes, fontsize=9)
        ax.yaxis.set_major_formatter(plt.NullFormatter())

    axes[-1].set_xlabel(r'Rest-frame wavelength [Å]', fontsize=10)
    fig.suptitle(title, fontsize=12)
    plt.savefig(f'plots/{fname}', dpi=130, bbox_inches='tight')
    plt.show()

plot_components(grid_hi, mean_hi, evecs_hi, evals_hi,
                'Fig 7.4 — PCA components, High-z QSOs (z>2)',
                'fig74_pca_highz.png')

plot_components(grid_lo, mean_lo, evecs_lo, evals_lo,
                'Fig 7.4 — PCA components, Low-z QSOs (0.5<z<1)',
                'fig74_pca_lowz.png')

## 4. Figure 7.5 — Eigenvalues

Top panel: individual eigenvalues (log-log scale).
Bottom panel: cumulative explained variance.

In [ ]:
def plot_eigenvalues(evals, evals_cs, title, fname):
    """Fig 7.5: eigenvalue spectrum and cumulative variance."""
    idx = np.arange(1, len(evals)+1)

    fig, axes = plt.subplots(2, 1, figsize=(6, 5))
    fig.subplots_adjust(hspace=0.08)

    # Individual eigenvalues
    axes[0].plot(idx, evals, 'k-o', ms=4)
    axes[0].set_xscale('log'); axes[0].set_yscale('log')
    axes[0].set_ylabel('Explained variance ratio')
    axes[0].xaxis.set_major_formatter(plt.NullFormatter())
    axes[0].grid(alpha=0.3)

    # Cumulative
    axes[1].semilogx(idx, evals_cs, 'k-o', ms=4)
    axes[1].set_xlabel('Eigenvalue number')
    axes[1].set_ylabel('Cumulative variance')
    axes[1].set_ylim(evals_cs[0]*0.9, 1.02)
    axes[1].grid(alpha=0.3)
    # Mark 90% and 95% levels
    for level, ls in [(0.90, '--'), (0.95, ':')]:
        axes[1].axhline(level, color='gray', lw=1, ls=ls,
                        label=f'{int(level*100)}%')
    axes[1].legend(fontsize=8)

    fig.suptitle(title, fontsize=11)
    plt.savefig(f'plots/{fname}', dpi=130, bbox_inches='tight')
    plt.show()

plot_eigenvalues(evals_hi, evals_cs_hi,
                 'Fig 7.5 — Eigenvalues, High-z QSOs (z>2)', 'fig75_evals_highz.png')
plot_eigenvalues(evals_lo, evals_cs_lo,
                 'Fig 7.5 — Eigenvalues, Low-z QSOs (0.5<z<1)', 'fig75_evals_lowz.png')

## 5. Figure 7.6 — Spectrum reconstruction

A single spectrum reconstructed with 0, 4, 8 and 20 PCA components.
Gray = original spectrum; black = reconstruction.

In [ ]:
def plot_reconstruction(grid, matrix, mean, evecs, evals_cs,
                        spec_idx, title, fname,
                        n_steps=[0, 4, 8, 20]):
    """
    Fig 7.6: progressive reconstruction of one spectrum.
    spec_idx selects which spectrum to reconstruct.
    """
    spec  = matrix[spec_idx]
    # Project spectrum onto eigenvectors
    coeff = evecs @ (spec - mean)   # shape (n_components,)

    fig, axes = plt.subplots(len(n_steps), 1, figsize=(10, 2.5*len(n_steps)),
                              sharex=True)
    fig.subplots_adjust(hspace=0)

    for ax, n in zip(axes, n_steps):
        recon = mean + (evecs[:n].T @ coeff[:n]) if n > 0 else mean
        ax.plot(grid, spec,  color='gray', lw=0.8, alpha=0.8)
        ax.plot(grid, recon, color='k',    lw=1.2)
        ax.set_ylabel(r'$F_\lambda$ (norm.)', fontsize=8)
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        if n == 0:
            label = 'mean only'
        else:
            label = f'mean + {n} components  '
            label += r'($\sigma^2_{\rm tot}$' + f' = {evals_cs[n-1]:.2f})'
        ax.text(0.02, 0.88, label, transform=ax.transAxes, fontsize=9)

    axes[-1].set_xlabel(r'Rest-frame wavelength [Å]', fontsize=10)
    fig.suptitle(title, fontsize=11)
    plt.savefig(f'plots/{fname}', dpi=130, bbox_inches='tight')
    plt.show()

# Choose a representative spectrum (index 5 — change as desired)
plot_reconstruction(grid_hi, matrix_hi, mean_hi, evecs_hi, evals_cs_hi,
                    spec_idx=5,
                    title='Fig 7.6 — Reconstruction, High-z QSO',
                    fname='fig76_recon_highz.png')

plot_reconstruction(grid_lo, matrix_lo, mean_lo, evecs_lo, evals_cs_lo,
                    spec_idx=5,
                    title='Fig 7.6 — Reconstruction, Low-z QSO',
                    fname='fig76_recon_lowz.png')

## 6. Side-by-side comparison: high-z vs low-z mean spectra

Shows the different spectral features captured by each sample.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(grid_hi, mean_hi, 'k', lw=1.2)
axes[0].set_title('Mean spectrum — High-z QSOs (z>2, rest-frame UV)')
axes[0].set_ylabel(r'$F_\lambda$ (norm.)')
# Mark main emission lines in UV
for lam, name in [(1216,'Ly-α'), (1549,'C IV'), (1909,'C III]')]:
    if grid_hi.min() < lam < grid_hi.max():
        axes[0].axvline(lam, color='C1', lw=0.8, ls='--', alpha=0.7)
        axes[0].text(lam+10, axes[0].get_ylim()[1]*0.85, name, fontsize=8, color='C1')

axes[1].plot(grid_lo, mean_lo, 'k', lw=1.2)
axes[1].set_title('Mean spectrum — Low-z QSOs (0.5<z<1, rest-frame optical)')
axes[1].set_ylabel(r'$F_\lambda$ (norm.)')
axes[1].set_xlabel(r'Rest-frame wavelength [Å]')
# Mark main emission lines in optical
for lam, name in [(2799,'Mg II'), (4861,'Hβ'), (4959,'[O III]'), (5007,'[O III]')]:
    if grid_lo.min() < lam < grid_lo.max():
        axes[1].axvline(lam, color='C0', lw=0.8, ls='--', alpha=0.7)
        axes[1].text(lam+30, axes[1].get_ylim()[1]*0.85, name, fontsize=8, color='C0')

plt.tight_layout()
plt.savefig('plots/mean_spectra_comparison.png', dpi=130, bbox_inches='tight')
plt.show()